In [1]:
# Install required libraries
!pip install google-generativeai python-dotenv pymupdf sentence-transformers faiss-cpu numpy

In [2]:
import pymupdf       # for reading PDFs
from sentence_transformers import SentenceTransformer   # to convert text into numeric vectors
import faiss                   # for fast similarity search
import numpy as np
import google.generativeai as genai   # for generating answers
from dotenv import load_dotenv   # to load the API key from .env
import os

C:\Users\Admin\AppData\Local\Temp\ipykernel_1572\2684872053.py:5: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai   # for generating answers


# Extract text from PDF

In [3]:
def extract_text(pdf_path):
    doc = pymupdf.open(pdf_path)      # here we open the PDF
    text = ""
    for page in doc:               # combine text from every page
        text += page.get_text()
    return text

text = extract_text("Placement Policy 2027_B.Tech_MCA.pdf")   # pdf path
print(len(text), "characters extracted")

8197 characters extracted


# Split text into chunks

In [4]:
# The whole text can't be processed at once, so we split it into smaller chunks
def chunk_text(text, chunk_size=300):
    words = text.split()       # Here we split text into words
    chunks = []
    for i in range(0, len(words), chunk_size):
        chunk = " ".join(words[i:i+chunk_size])   # now we group every 300 words together
        chunks.append(chunk)
    return chunks

chunks = chunk_text(text)
print(len(chunks), "chunks created")

5 chunks created


# Convert chunks into embeddings

In [5]:
# Load a model that converts text into vectors (numbers)
model = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = model.encode(chunks)   # create a vector for each chunk

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

# Build FAISS index

In [6]:
# FAISS is a fast tool for finding similar vectors
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)   # simple distance-based search index
index.add(np.array(embeddings)) # add all chunk embeddings to the index

# Retrieve relevant chunks for a question

In [7]:
def get_relevant_chunks(question, k=3):
    q_embedding = model.encode([question])        # create a vector for the question
    distances, indices = index.search(np.array(q_embedding), k)   # find the top-k most similar chunks
    return [chunks[i] for i in indices[0]]                      # return their text

# Gemini setup (API key from .env)

In [8]:
load_dotenv()   # load the .env file

api_key = os.getenv("GEMINI_API_KEY")  
genai.configure(api_key=api_key)

gen_model = genai.GenerativeModel("gemini-3.6-flash")

# Function to generate the answer

In [15]:
def ask_question(question):
    relevant_chunks = get_relevant_chunks(question)   # get related chunks from the PDF
    context = "\n".join(relevant_chunks)                # combine them into one block

    prompt = f"""Answer the question based on the context below and use my name Faisal Khan for answer.
Context: {context}
Question: {question}
Answer:"""

    response = gen_model.generate_content(prompt)     # get the answer from the AI
    return response.text

# Testing

In [16]:
print(ask_question("what is this pdf about"))

Faisal Khan, based on the provided context, this document is about the **KIET University Placement Policy for the 2027 Batch (B.Tech./MCA)**. 

It outlines the official rules, procedures, and guidelines regarding campus placements for students, including:
* The **"One Student–One Offer" policy** and eligibility rules for further opportunities.
* Definitions and criteria for **Dream Offers** (CTC ₹9.00 LPA or above) and **Super Dream Offers** (CTC ₹16.00 LPA or above).
* Rules for early joining, internships, and mandatory company pre-joining modules (e.g., TCS Xplore, Wipro TalentNext).
* Mandatory preparation and pre-assessment guidelines via the IAMNeo platform.
* Strict codes of conduct, discipline, document verification policies, and communication protocols with recruiters.


In [17]:
print(ask_question("what happen if i got 1 offer from a company"))

Hello Faisal Khan, based on the placement policy provided, if you get one offer from a company, the following rules and conditions will apply to you:

1. **One Student–One Offer Policy:** KIET follows a strictly enforced "One Student–One Offer" policy. As soon as you receive an offer or selection, you must on your own stop appearing in further drives. 

2. **Eligibility for Further Drives (Dream / Super Dream Companies):** After being offered a job, you are generally allowed to participate further **only** in Super Dream or Dream Company placement processes under specific conditions:
   * **CTC Requirements:** If your offer CTC is between ₹7.00 LPA and ₹8.99 LPA, you will be eligible for a Dream Offer opportunity only if the new CTC is at least **1.50 times higher** than your previous offer.
   * **Equal/Higher Package Restriction:** You will not be eligible for a Dream company if your existing offer package is equal to or higher than that Dream company's package.
   * **Post-Joining L

In [18]:
print(ask_question("Dress code"))

Hello Faisal Khan, according to the guidelines, the dress code requires students to wear proper business formals for every recruitment process, which includes:

* **Trouser:** Blue/Black
* **Shirt:** White/Light color
* **Tie:** KIET Tie
* **Shoes:** Black


In [20]:
print(ask_question("Tell me about dream offer"))

Hello Faisal Khan, based on the provided placement policy context, here are the details regarding a **Dream Offer / Dream Company**:

* **Definition:** Companies offering a package of **₹9.00 LPA or above** are generally designated as Dream Companies. However, CRPC has the right to make decisions on this categorization based on prevailing conditions and the availability of students.
* **Eligibility Criteria & Rules:**
  * **Previous CTC Condition:** Students selected with a CTC from Rs. 7.00 LPA to Rs. 8.99 LPA will be eligible for a Dream Offer as their next opportunity if the CTC is **1.50 times higher** than their previous offer.
  * **Package Comparison:** A student is **not eligible** for a Dream company if he/she has already been selected in a company of equal or higher package than that Dream company.
  * **Participation after an Offer:** After receiving a job offer from any company, a student is allowed to participate further in the Super Dream / Dream Company placement process